# QoR history: how quality of results changes over runs

This notebook runs the FIFO of the examples through `yosys_fpga+nextpnr` for an ECP5 device four
times. Each run has other settings: three clock periods and a second seed. Then it reads the
records of these runs and builds a table. At the end, it plots `fmax` over the time of each run.

The notebook needs Yosys, `nextpnr-ecp5`, pandas and Plotly.

In [ ]:
import os
from pathlib import Path

import xeda

EXAMPLES = Path(os.environ.get("XEDA_EXAMPLES", Path.cwd().parent / "examples")).resolve()
if not EXAMPLES.is_dir():
    raise FileNotFoundError(f"No examples in {EXAMPLES}. Set XEDA_EXAMPLES to the examples directory.")

WORK = Path("notebook-output").resolve()
ws = xeda.Workspace(WORK / "xeda_run", hashed_run_dirs=True)

## Make some runs

The runs differ in the clock period of `yosys_fpga` and in the seed of `nextpnr`. A workspace with
`hashed_run_dirs=True` keeps the run of each configuration in a directory of its own. Xeda reuses
a run that is already there, so you can run this cell again.

In [ ]:
from xeda import Design, Spec

design = Design.load(EXAMPLES / "sv/fifo/fifo.xeda.yaml")


def spec(period: str, seed: int) -> Spec:
    return Spec(
        "yosys_fpga+nextpnr",
        design,
        settings={
            "yosys_fpga": {"fpga.part": "LFE5U-85F-6BG381C", "clock.period": period},
            "nextpnr": {"lpf_allow_unconstrained": True, "seed": seed},
        },
    )


for period, seed in (("12 ns", 1), ("10 ns", 1), ("8 ns", 1), ("10 ns", 2)):
    result = ws.run(spec(period, seed))
    print(period, seed, result.outcome.value)

## Read the past runs

`ws.records` reads the records from the run root. It starts no tool and takes no lock. The filters
are `design_name`, `flow` and `target`. A filter that you leave out matches everything. This
listing has the records of `nextpnr`. The code sorts them by `finished_at`, the time when the run
ended.

In [ ]:
history = ws.records(design_name="fifo", flow="nextpnr")
records = sorted(history, key=lambda r: r.finished_at)
print(len(records), "records,", len(history.problems), "problems")

## Make a table

Each `RunRecord` has the `settings` of its node and its `metrics`. The table keeps only the
settings that differ between the runs. Settings that are equal in all runs do not explain a
change. A run that did not pass has `None` for a metric that its record does not hold.

In [ ]:
from collections.abc import Mapping

import pandas as pd


def flatten(table, prefix=""):
    """The leaves of a nested table, by dotted name."""
    for key, value in table.items():
        if isinstance(value, Mapping):
            yield from flatten(value, f"{prefix}{key}.")
        else:
            yield f"{prefix}{key}", value


settings = [dict(flatten(r.settings)) for r in records]
names = {name for s in settings for name in s}
differing = sorted(n for n in names if len({str(s.get(n)) for s in settings}) > 1)

df = pd.DataFrame(
    [
        {
            "flow": r.flow,
            "design": r.design_name,
            **{n: s.get(n) for n in differing},
            "finished_at": r.finished_at,
            "outcome": r.outcome.value,
            "fmax": r.metrics.get("fmax"),
            "lut": r.metrics.get("lut"),
        }
        for r, s in zip(records, settings)
    ]
)
df.to_csv(WORK / "history.csv", index=False)
df

The table is also in `notebook-output/history.csv`. This cell prints the number of rows.

In [ ]:
print(len(df))

## Plot `fmax` over time

The plot uses Plotly and shows in the notebook. It does not export an image. The unit comes from
the record.

In [ ]:
import plotly.graph_objects as go

unit = records[0].metric_info["fmax"].unit

fig = go.Figure()
fig.add_scatter(
    x=df.finished_at,
    y=df.fmax,
    mode="lines+markers",
    text=[", ".join(f"{n}={row[n]}" for n in differing) for _, row in df.iterrows()],
    hovertemplate="%{text}<br>%{y:.1f}<extra></extra>",
)
fig.update_layout(template="plotly_white", xaxis_title="finished at", yaxis_title=f"fmax ({unit})")
fig.show()

## Two cases with nothing to show

A run root with no runs gives an empty listing. It is not an error.

In [ ]:
listing = xeda.Workspace(WORK / "empty").records(design_name="fifo")
if not listing.records:
    print("no runs yet: run the cells above first, or open a workspace on a run root that has runs")

`RunRecord.metric` reads one metric by its dotted name. A name that no metric has raises
`KeyError`. The message is the name.

In [ ]:
try:
    records[0].metric("nosuch.metric")
except KeyError as error:
    print(error.args[0])